# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

../../../../../../assets/03-wine-quality/inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# Práctica A — lectura del CSV con csv.DictReader
def read_csv_wines(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


filas_csv = read_csv_wines(DATA_PATH)
row = filas_csv[0]

print("Número de filas:", len(filas_csv))
print("Columnas:", list(row.keys()))
print("Primera fila:", row)

Número de filas: 5
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Primera fila: {'sample_id': 'red-001', 'fixed_acidity': '7.4', 'volatile_acidity': '0.7', 'citric_acid': '0.0', 'residual_sugar': '1.9', 'chlorides': '0.076', 'free_sulfur_dioxide': '11.0', 'total_sulfur_dioxide': '34.0', 'density': '0.9978', 'ph': '3.51', 'sulphates': '0.56', 'alcohol': '9.4'}


### Preguntas (respuestas también en `Respuestas_teoría.md`)

1. ¿Qué campo identifica la muestra pero no es una característica del modelo? **`sample_id`**. Identifica la fila, pero el modelo no lo recibe.
2. ¿Cuántas características numéricas hay? **11** (todas las columnas excepto `sample_id`).
3. ¿Qué columna extra añadirías para probar el contrato? **`unexpected_field`** (o `quality`, la etiqueta, que no debe llegar en inferencia).
4. ¿Qué valor de `ph` sería claramente inválido? **`ph = 99.0`** (fuera del rango 2.5–4.5).

### Entrenamiento, artefacto y datos nuevos

| Pieza | Qué es | En este ejercicio | ¿La construimos? |
|---|---|---|---|
| **Datos de entrenamiento** | CSV histórico *con* la etiqueta `quality` | *Red Wine Quality (UCI)* completo, solo lo usa el profesorado | No; no se versiona en Git |
| **Artefacto** | Modelo ya entrenado y serializado | `models/wine_quality_classifier.joblib` (`estimator`, `feature_names`, `model_version`) | No; se carga, no se reentrena |
| **Datos nuevos** | Filas *sin* etiqueta sobre las que se predice | `assets/03-wine-quality/inference_samples.csv` (5 filas) | No; los validamos y preprocesamos |

```text
datos de entrenamiento -> entrenamiento -> artefacto .joblib
fila nueva -> contrato -> preprocesado -> artefacto cargado -> predicción
```

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características. Unidades según la documentación del dataset (Cortez et al., 2009, `winequality.names`).

| Campo | Tipo | Unidad | Límite razonable |
|---|---|---|---|
| `fixed_acidity` | float | g/dm³ (ácido tartárico) | 0–20 |
| `volatile_acidity` | float | g/dm³ (ácido acético) | 0–2 |
| `citric_acid` | float | g/dm³ | 0–2 |
| `residual_sugar` | float | g/dm³ | 0–20 |
| `chlorides` | float | g/dm³ (cloruro sódico) | 0–1 |
| `free_sulfur_dioxide` | float | mg/dm³ | 0–100 |
| `total_sulfur_dioxide` | float | mg/dm³ | 0–300 |
| `density` | float | g/cm³ | 0.98–1.01 |
| `ph` | float | adimensional (escala pH) | 2.5–4.5 |
| `sulphates` | float | g/dm³ (sulfato potásico) | 0–3 |
| `alcohol` | float | % vol. | 5–20 |

`sample_id` (str) **no** está en el contrato del modelo: se separa antes de validar.

**Política de ausentes:** los 11 campos son obligatorios y no se imputa nada en inferencia. Una columna que falta → error `missing`; una celda vacía (`""`) → error `float_parsing`. En ambos casos la fila se rechaza y el script se detiene sin generar salida.

In [3]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    # TODO: declara las 11 características con Field(ge=..., le=...).
    model_config = ConfigDict(extra = "forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)
    

# TODO: separa sample_id de la fila y valida solo las características del modelo.
model_row = row.copy()
sample_id = model_row.pop("sample_id")
request = WineQualityRequestPydantic.model_validate(model_row)
print(request)


fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4


### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.
- (Ausentes) Falta la columna `alcohol` / celda `alcohol` vacía.

Antes de ejecutar, escribo qué error espero:

1. **Fila válida** → no hay error; se crea la petición.
2. **`unexpected_field`** → `ValidationError` de tipo `extra_forbidden` (el contrato prohíbe columnas extra).
3. **`ph = 99.0`** → `ValidationError` de tipo `less_than_equal` (supera el límite 4.5).
4. **Falta `alcohol`** → `ValidationError` de tipo `missing`.
5. **`alcohol = ""`** → `ValidationError` de tipo `float_parsing`.

In [4]:
valid_request = WineQualityRequestPydantic.model_validate(model_row)
print("Fila válida:")
print(valid_request)

sin_alcohol = {k: v for k, v in model_row.items() if k != "alcohol"}

invalid_cases = {
    "Campo extra": ({**model_row, "unexpected_field": "no permitido"}, "extra_forbidden"),
    "ph fuera de rango": ({**model_row, "ph": 99.0}, "less_than_equal"),
    "Falta alcohol": (sin_alcohol, "missing"),
    "alcohol vacío": ({**model_row, "alcohol": ""}, "float_parsing"),
}

for caso, (datos, error_esperado) in invalid_cases.items():
    try:
        WineQualityRequestPydantic.model_validate(datos)
        raise AssertionError(f"{caso}: se esperaba un error y no lo hubo")
    except ValidationError as error:
        tipo = error.errors()[0]["type"]
        print(f"\n{caso}: esperado={error_esperado} obtenido={tipo}")
        print(error)
        assert tipo == error_esperado

Fila válida:
fixed_acidity=7.4 volatile_acidity=0.7 citric_acid=0.0 residual_sugar=1.9 chlorides=0.076 free_sulfur_dioxide=11.0 total_sulfur_dioxide=34.0 density=0.9978 ph=3.51 sulphates=0.56 alcohol=9.4

Campo extra: esperado=extra_forbidden obtenido=extra_forbidden
1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='no permitido', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden

ph fuera de rango: esperado=less_than_equal obtenido=less_than_equal
1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=99.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Falta alcohol: esperado=missing obtenido=missing
1 validation error for WineQualityRequestPydantic
alcohol
  Field required [type=missing, input_value={'fixed_acidity':

## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.


1. no garantiza que se establezcan unos rangos 


In [5]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float


valid_dataclass = WineQualityRequestDataclass(**request.model_dump())
print(valid_dataclass)

invalid_ph_data = {**request.model_dump(), "ph": 99.0}
invalid_ph_dataclass = WineQualityRequestDataclass(**invalid_ph_data)
print(invalid_ph_dataclass)

WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=99.0, sulphates=0.56, alcohol=9.4)


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en la validacion de una fila del csv porque me asegura establecer unos límites y que se cumplan . Usaría `dataclass` en situaciones donde no haya que validar nada.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [6]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
)

vector = [getattr(request, name) for name in FEATURE_NAMES]
print(vector)
 

[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?. Sí
- ¿El clasificador detectaría necesariamente el error? No necesariamente, pero sí por los límites 
- ¿Cómo evitarías que cada script definiera un orden distinto? Haciendo un vector de características constante 

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?
- ¿Qué categoría se ha predicho?
- ¿Con qué confianza?
- ¿Qué versión de modelo y preprocesado se utilizaron?

In [7]:
QualityBand = Literal["needs_review", "acceptable", "excellent"]

class WineQualityPredictionPydantic(BaseModel):
    sample_id: str
    quality_band: QualityBand
    confidence: float = Field(ge=0, le=1)
    model_version: str
    preprocessing_version: str

# Ejemplo  
prediction = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="acceptable",
    confidence=0.87,
    model_version="wine-quality-rf-v1",
    preprocessing_version="wine-red-features-v1",
)
print(prediction)

sample_id='red-001' quality_band='acceptable' confidence=0.87 model_version='wine-quality-rf-v1' preprocessing_version='wine-red-features-v1'


## 7. Criterios de aceptación para el script de la clase 2

1. **Fila válida → predicción:** cada fila de `inference_samples.csv` pasa el contrato y produce un vector de 11 `float` en el orden de `FEATURE_NAMES`.
2. **Columna desconocida → el script se detiene:** una fila con `unexpected_field` (o con un campo ausente) se rechaza con `ValidationError`; el script debe salir con código 2 y sin CSV de salida.
3. **Trazabilidad:** toda salida incluye `sample_id`, `quality_band`, `confidence`, `model_version` y `preprocessing_version`.

La celda siguiente los comprueba de forma ejecutable.

In [8]:
# Criterio 1: todas las filas válidas generan un vector de 11 floats en orden estable
for fila in filas_csv:
    datos = {k: v for k, v in fila.items() if k != "sample_id"}
    req = WineQualityRequestPydantic.model_validate(datos)
    vec = [getattr(req, name) for name in FEATURE_NAMES]
    assert len(vec) == 11 and all(isinstance(x, float) for x in vec)
    assert vec[FEATURE_NAMES.index("alcohol")] == float(fila["alcohol"])
print("Criterio 1 OK:", len(filas_csv), "filas válidas")

# Criterio 2: una columna desconocida detiene la validación
try:
    WineQualityRequestPydantic.model_validate({**model_row, "unexpected_field": 1})
    raise AssertionError("La columna extra debería rechazarse")
except ValidationError as error:
    assert error.errors()[0]["type"] == "extra_forbidden"
print("Criterio 2 OK: columna extra rechazada")

# Criterio 3: la salida conserva identificador y las dos versiones
columnas_salida = set(WineQualityPredictionPydantic.model_fields)
assert {"sample_id", "model_version", "preprocessing_version"} <= columnas_salida
assert prediction.model_version and prediction.preprocessing_version
print("Criterio 3 OK:", sorted(columnas_salida))

Criterio 1 OK: 5 filas válidas
Criterio 2 OK: columna extra rechazada
Criterio 3 OK: ['confidence', 'model_version', 'preprocessing_version', 'quality_band', 'sample_id']


## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.